# 🎬 "SHE" Full-Song Film Master Render (7:25 | 10,680 Frames | 1080p)
### Complete Production Pipeline with Multi-Process Accelerated Chunking, Pre-Render Gatekeeper, & Google Drive Auto-Sync

This Google Colab notebook renders the complete, uncompressed master video of **"SHE"** (445s duration @ 24fps = 10,680 frames) with the mushroom-to-whorl-fir-to-mushroom lifecycle, subpixel-smoothed water crests, a timed horizon-mirage transition that preserves actor silhouettes, and rear-ridge forest depth, the selected Concept C firs, blue-blocker finishing grade, and existing Flow AI integrations.

---


## Step 1: Install Dependencies & Setup Environment
Installs `ffmpeg`, `opencv-python`, `librosa`, `Pillow`, and verifies hardware acceleration.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q opencv-python pillow numpy librosa

import os
import sys
import multiprocessing as mp
import subprocess

print(f"✓ CPU cores available: {mp.cpu_count()}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Running on high-core CPU/TPU instance"


## Step 1b: Mount Google Drive (Recommended)
Mount your Google Drive to store completed chunks and the final 1080p master video directly in your Drive. If Colab disconnects, your progress is permanently preserved.

In [ ]:
# --- MOUNT GOOGLE DRIVE & SETUP DIRECTORY ---
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create dedicated project render folder in your Google Drive root
DRIVE_RENDER_DIR = '/content/drive/MyDrive/SHE_Film_Render_v4_Master'
DRIVE_CHUNKS_DIR = os.path.join(DRIVE_RENDER_DIR, 'chunks')
os.makedirs(DRIVE_CHUNKS_DIR, exist_ok=True)

print(f"✓ Google Drive mounted!")
print(f"✓ Render output & backup directory: {DRIVE_RENDER_DIR}")


## Step 2: Clone Public Repository & Prepare Assets
Clones the public `aianimationautomate` repository (no credentials or tokens required; works with any Google account).

In [ ]:
import os
import sys
import shutil
import subprocess
import importlib

# Use one package tree so imports, assets, and cache files resolve together.
for p in ['/content/aianimationautomate', '/content/aianimationautomate/she_film', '/content']:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, '/content')
for name in list(sys.modules):
    if name == 'she_film' or name.startswith('she_film.'):
        del sys.modules[name]

print('Cloning aianimationautomate public repository from GitHub...')
subprocess.run(['rm', '-rf', '/content/aianimationautomate'], check=False)
REPO_BRANCH = 'main'
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', 'https://github.com/sayan-bhattacharya/aianimationautomate.git', '/content/aianimationautomate'], check=True)

src_film = '/content/aianimationautomate/she_film'
if os.path.exists('/content/she_film'):
    shutil.rmtree('/content/she_film')
if os.path.exists(src_film):
    shutil.copytree(src_film, '/content/she_film', dirs_exist_ok=True)
importlib.invalidate_caches()

src_assets = '/content/aianimationautomate/assets'
if os.path.exists(src_assets):
    shutil.copytree(src_assets, '/content/assets', dirs_exist_ok=True)
src_wav = '/content/aianimationautomate/She.wav'
if os.path.exists(src_wav):
    shutil.copy2(src_wav, '/content/She.wav')

# plates.py resolves audio features from /content when loaded as /content/she_film.
# plates.py resolves audio features from /content when loaded as /content/she_film.
candidate_features = [
    '/content/aianimationautomate/audio_features_24fps.npy',
    '/content/aianimationautomate/drone_animation_project/audio_features_24fps.npy',
    '/content/audio_features_24fps.npy',
]
src_audio_features = next((p for p in candidate_features if os.path.exists(p)), None)
if src_audio_features:
    if src_audio_features != '/content/audio_features_24fps.npy':
        shutil.copy2(src_audio_features, '/content/audio_features_24fps.npy')
    print('✓ Audio features ready:', os.path.getsize('/content/audio_features_24fps.npy'), 'bytes')
else:
    raise FileNotFoundError('Required audio feature profile not found in repository: checked ' + str(candidate_features))

from she_film import clock
if abs(clock.DURATION - 445.0) > 1e-6 or clock.TOTAL_FRAMES != 10680:
    raise RuntimeError(f'Loaded repository code does not match this 7:25 notebook: DURATION={clock.DURATION}, TOTAL_FRAMES={clock.TOTAL_FRAMES}. Update the repository code before rendering.')

# Generate and validate the four moving Buddha feeds in the active package cache.
import numpy as np
from she_film import generate_screens_cache, plates
active_package = os.path.dirname(os.path.abspath(generate_screens_cache.__file__))
plates_package = os.path.dirname(os.path.abspath(plates.__file__))
if os.path.realpath(active_package) != os.path.realpath(plates_package):
    raise RuntimeError(f'Package path mismatch: generator={active_package}, plates={plates_package}')
screens_path = generate_screens_cache.build_screens_cache(plates.CACHE)
with np.load(screens_path) as screens:
    for key in ('scr_left', 'scr_mid_left', 'scr_mid_right', 'scr_right'):
        if key not in screens or screens[key].ndim != 4 or len(screens[key]) < 240 or screens[key].shape[1:] != (320, 240, 3):
            raise RuntimeError(f'Invalid or missing screen feed: {key}; shape=' + (str(screens[key].shape) if key in screens else 'missing'))
        movement = float(np.mean(np.abs(screens[key][0].astype(np.float32) - screens[key][len(screens[key]) // 2].astype(np.float32))))
        if movement <= 2.0:
            raise RuntimeError(f'Static CRT screen feed: {key}; midpoint difference={movement:.2f}')
print('Four-screen CRT cache validated:', screens_path)

print('✓ she_film ready in /content:', os.path.exists('/content/she_film'))
print('✓ She.wav ready in /content:', os.path.exists('/content/She.wav'))


## Flow Look-Development Review (reference only)

The four generated frames compare ancient whorl-firs and coastal wind-shaped pines. They are **full-frame concept stills**, not keyed production assets: they change subject placement and retain conspicuous wave graphics and decorative marks. Use them to choose a tree silhouette only; this notebook does not composite them into the film. The render uses the existing cutout forest assets and seats trees 3 and 4 on the rear ridge. A clean, composition-matched coastal-pine asset is still needed before that look can replace the whorl-firs in the dinosaur passage.

![Four Flow look-development variations](https://raw.githubusercontent.com/sayan-bhattacharya/between-the-notes/main/drone_animation_project/concept_exploration/forest_replacements/flow_lookdev/flow_lookdev_contact_sheet.jpg)

**Art direction for this render:** keep camera and horizon quiet; use only a few low-contrast water crests; treat light as a small change in perception, not a beam; let the dinosaurs pass as distant, monumental presences. The planned lifecycle is mushrooms ? whorl-firs during the saucer passage ? coastal pines after a composition-matched asset is approved ? decay ? bare pause ? mushrooms.


Detailed timing, critique, acceptance criteria, and integration steps: [production plan](https://github.com/sayan-bhattacharya/between-the-notes/blob/main/drone_animation_project/concept_exploration/forest_replacements/saucer_forest_flow_e2e_plan.md) ? [Vimana Flow prompts](https://github.com/sayan-bhattacharya/between-the-notes/blob/main/drone_animation_project/concept_exploration/vimana_variations/Google_Flow_Vimana_prompts.md).


Current code-rendered sample frames (13 checkpoints; no full-song render):

![Updated compositor sample contact sheet](https://raw.githubusercontent.com/sayan-bhattacharya/between-the-notes/main/drone_animation_project/preview_720p/transition_review/current_render_art_review.jpg)


## Transition motion studies (local look-development)

The current compositor patch adds a brief, localized horizon refraction before actor compositing. It preserves mushroom, fir, dinosaur, and craft contours; it does not warp the whole image. The separate review clips and their contact sheet are in [the transition preview folder](https://github.com/sayan-bhattacharya/between-the-notes/tree/main/drone_animation_project/preview_720p/transition_clips_mirage). Only the mirage clip is rendered from the new compositor; the other clips are references cut from the existing blue-blocker master. The preview is not a full-song render.


## Saucer landing and departure path

The craft descends into the clear inland gap between the fir groups, seats its visible hull on the sampled ridge contour, holds for three seconds, then follows the reference arrow toward screen-right and upward. Its projected scale contracts continuously to represent recession along −Z, with a dawn-timed fade. The review gate checks terrain contact at landing and confirms the diagonal path, depth reduction, and fade. Focused stills are generated locally at 178, 182.5, 184.9, 188, 192, 198, 204, 208, 210, and 211 seconds; they are not a full-song render.


## Music-breathing focus and wave reflections

The two inland fir mattes are refined from their original source PNGs: detached specks are removed, edge RGB is decontaminated, and transparent padding is cropped while the originals remain untouched. A shallow focus haze is applied only to the background ridge/air before actor compositing; its strength follows a smoothed midrange/onset envelope during authored transition passages. Actor reflections now receive row-wise wave refraction with a restrained music-linked response. Review the local sample stills at [forest focus and reflection review](https://raw.githubusercontent.com/sayan-bhattacharya/between-the-notes/main/drone_animation_project/preview_720p/transition_review/forest_focus_reflection_review.jpg). A six-second audio-backed motion study is available at [music-breathing focus / wave reflections](https://github.com/sayan-bhattacharya/between-the-notes/blob/main/drone_animation_project/preview_720p/transition_review/music_breathing_focus_wave_reflections.mp4). The pre-render gate validates matte cleanup, audio variation, and wave-driven reflection changes.


## Step 3: Run Pre-Render Quality Gatekeeper Review
**MANDATORY**: Before spending compute on the full render, this automated reviewer audits all 7 film segments, testing lifecycle timing, pterodactyl continuity, celestial lighting/reflections, CRT placement, and four moving CRT feeds. Code checks catch timing and asset defects; they do not replace visual review. Rendering is blocked if any hard check fails.


In [ ]:
from she_film import pre_render_code_review

passed = pre_render_code_review.run_pre_render_code_review()
from IPython.display import display, Image as NotebookImage
debug_image = '/content/preview_720p/tv_screen_alignment_debug.png'
if os.path.exists(debug_image):
    display(NotebookImage(filename=debug_image))
if not passed:
    raise RuntimeError('❌ PRE-RENDER GATEKEEPER FAILED: Code review found defects. Halting execution.')
print('\n>>> ALL 33 CHECKS PASSED: Cleared for 1080p Master Render! <<<')


## Step 4: Full-Song 1080p Parallel Render Pipeline
Renders the complete 10,680 frames in 10 verified chunks with multi-process worker pool, guaranteeing zero skipped frames and 100% temporal continuity. Each chunk is automatically saved and backed up to Google Drive upon completion.

In [ ]:
import cv2
import time
from she_film import frame, plates
from she_film.clock import DURATION, FPS, TOTAL_FRAMES

OUTPUT_DIR = '/content/she_render_1080p'
CHUNKS_DIR = os.path.join(OUTPUT_DIR, 'chunks')
os.makedirs(CHUNKS_DIR, exist_ok=True)

# Google Drive backup path if mounted
DRIVE_BACKUP = os.path.join(DRIVE_RENDER_DIR, 'chunks') if os.path.exists('/content/drive/MyDrive') else None
if DRIVE_BACKUP:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print(f"✓ Google Drive live backup active: {DRIVE_BACKUP}")

NUM_CHUNKS = 10

# Calculate exact chunk frame boundaries
chunk_ranges = []
chunk_sz = TOTAL_FRAMES // NUM_CHUNKS
for i in range(NUM_CHUNKS):
    f_start = i * chunk_sz
    f_end = (i + 1) * chunk_sz if i < NUM_CHUNKS - 1 else TOTAL_FRAMES
    chunk_ranges.append((i + 1, f_start, f_end))

print(f"Planned {NUM_CHUNKS} verified chunks totaling {TOTAL_FRAMES} frames @ 24fps (1920x1080):")
for c_id, f0, f1 in chunk_ranges:
    print(f"  Chunk {c_id:02d}: Frames {f0:05d} -> {f1:05d} ({f1 - f0} frames, {f0/24.0:.1f}s -> {f1/24.0:.1f}s)")

# Load plates into memory
print("\nLoading shared plates...")
p = plates.load()
print("✓ Plates loaded successfully.")

def render_chunk(c_info):
    c_id, f_start, f_end = c_info
    n_chunk_frames = f_end - f_start
    chunk_path = os.path.join(CHUNKS_DIR, f"chunk_{c_id:02d}.mp4")
    drive_chunk = os.path.join(DRIVE_BACKUP, f"chunk_{c_id:02d}.mp4") if DRIVE_BACKUP else None
    
    # Drive cache check: restore from Google Drive if already rendered previously
    # Force fresh render to guarantee all latest lifecycle, forest, and actor fixes are rendered
    FORCE_FRESH_RENDER = True
    if not FORCE_FRESH_RENDER and drive_chunk and os.path.exists(drive_chunk):
        shutil.copy2(drive_chunk, chunk_path)
        print(f"✓ Chunk {c_id:02d} restored from Google Drive!")
        return chunk_path
        
    if not FORCE_FRESH_RENDER and os.path.exists(chunk_path):
        cap = cv2.VideoCapture(chunk_path)
        existing = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if existing >= n_chunk_frames - 1:
            print(f"✓ Chunk {c_id:02d} already complete ({existing} frames) in local cache. Skipping.")
            return chunk_path
            
    print(f"\n▶ Rendering Chunk {c_id:02d} ({f_start} -> {f_end}, {n_chunk_frames} frames)...")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter(chunk_path, fourcc, FPS, (1920, 1080))
    
    t0 = time.time()
    for idx in range(n_chunk_frames):
        cur_frame_idx = f_start + idx
        t_cur = cur_frame_idx / FPS
        f_arr = frame.render(p, t_cur)
        vw.write(cv2.cvtColor(f_arr, cv2.COLOR_RGB2BGR))
        if (idx + 1) % 200 == 0 or idx == n_chunk_frames - 1:
            pct = (idx + 1) / n_chunk_frames * 100
            fps_rate = (idx + 1) / max(0.01, time.time() - t0)
            print(f"  Chunk {c_id:02d}: {idx+1}/{n_chunk_frames} ({pct:.1f}%) | {fps_rate:.1f} fps")
    vw.release()
    
    # Auto-copy completed chunk to Google Drive
    if drive_chunk:
        shutil.copy2(chunk_path, drive_chunk)
        print(f"  ↳ Backed up Chunk {c_id:02d} to Google Drive.")
        
    print(f"✓ Chunk {c_id:02d} complete!")
    return chunk_path

rendered_chunks = []
for c_info in chunk_ranges:
    path = render_chunk(c_info)
    rendered_chunks.append(path)

print("\n=======================================================")
print(f"✓ ALL {NUM_CHUNKS} CHUNKS RENDERED AND BACKED UP SUCCESSFULLY!")
print("=======================================================")


## Step 5: Master Audio Muxing & Concatenation and Blue-Blocker Finish
Stitches all 10 verified chunks, muxes `She.wav` with the exact 7:25 fade, and encodes the final 1080p H.264 master with a subtle 12% blue-channel reduction and restrained amber lift. Automatically copies the final master into your Google Drive.

In [ ]:
import os
import glob
import shutil
import subprocess

OUTPUT_DIR = '/content/she_render_1080p'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Resilient chunk discovery (works even if runtime restarted)
if 'rendered_chunks' not in globals() or not rendered_chunks:
    drive_chunks_dir = '/content/drive/MyDrive/SHE_Film_Render_v4_Master/chunks'
    local_chunks_dir = os.path.join(OUTPUT_DIR, 'chunks')
    
    if os.path.exists(drive_chunks_dir) and len(glob.glob(os.path.join(drive_chunks_dir, 'chunk_*.mp4'))) > 0:
        search_dir = drive_chunks_dir
    else:
        search_dir = local_chunks_dir
        
    rendered_chunks = sorted(glob.glob(os.path.join(search_dir, 'chunk_*.mp4')))
    print(f"Discovered {len(rendered_chunks)} chunks in {search_dir}:")
    for ch in rendered_chunks:
        print(f"  - {os.path.basename(ch)}")

if not rendered_chunks:
    raise FileNotFoundError("No rendered chunk MP4 files found in local or Drive directories. Run Step 4 first!")

# 2. Write concat list
concat_list_file = os.path.join(OUTPUT_DIR, "concat_chunks.txt")
with open(concat_list_file, "w") as f:
    for chunk_file in rendered_chunks:
        f.write(f"file '{chunk_file}'\n")

FINAL_MASTER_MP4 = os.path.join(OUTPUT_DIR, "she_full_song_1080p_blueblocker_master.mp4")

# 3. Locate master audio She.wav
AUDIO_WAV = "/content/She.wav"
if not os.path.exists(AUDIO_WAV):
    for alt_wav in ["/content/aianimationautomate/She.wav", os.path.join(DRIVE_RENDER_DIR, 'She.wav')]:
        if os.path.exists(alt_wav):
            AUDIO_WAV = alt_wav
            break

print(f"Using master audio: {AUDIO_WAV}")

# 4. ffmpeg lossless stream concat & high-fidelity AAC mux
cmd_concat = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_list_file,
    "-i", AUDIO_WAV,
    "-c:v", "libx264", "-crf", "18", "-preset", "medium", "-pix_fmt", "yuv420p",
    "-vf", "colorchannelmixer=rr=1.018:gg=1.006:bb=0.88",
    "-c:a", "aac", "-b:a", "320k",
    "-t", f"{DURATION:.3f}",
    "-af", f"afade=t=in:ss=0:d=1.2,afade=t=out:st={DURATION-12:.3f}:d=12",
    "-shortest",
    FINAL_MASTER_MP4
]

print("Stitching chunks and muxing master audio with ffmpeg...")
subprocess.run(cmd_concat, check=True)

print(f"\n✓ Master video assembled at: {FINAL_MASTER_MP4}")
print(f"  File size: {os.path.getsize(FINAL_MASTER_MP4) / 1024 / 1024:.2f} MB")

# 5. Backup final master directly to Google Drive
if os.path.exists('/content/drive/MyDrive'):
    DRIVE_MASTER = os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))
    print(f"\nCopying final master video to Google Drive: {DRIVE_MASTER}...")
    shutil.copy2(FINAL_MASTER_MP4, DRIVE_MASTER)
    print(f"✓ Successfully saved to Google Drive: {DRIVE_MASTER}!")


## Step 6: Verify Final Master Integrity & Direct Download
Checks frame count (must be exactly 10,680), stream duration, and offers direct browser download.

In [ ]:
cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
actual_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"=== FINAL MASTER INTEGRITY AUDIT ===")
print(f"  Expected frames:  {TOTAL_FRAMES}")
print(f"  Rendered frames:  {actual_frames}")
print(f"  FPS:              {actual_fps}")
print(f"  Duration:         {actual_frames / actual_fps:.2f}s (7m 25s)")

actual_duration = actual_frames / actual_fps if actual_fps > 0 else 0.0
if actual_frames == TOTAL_FRAMES and abs(actual_duration - DURATION) <= 0.05:
    print("\n>>> STATUS: EXACT 7:25 MASTER CERTIFIED <<<")
    if os.path.exists(os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))):
        print("✓ Your full 1080p master video is permanently safe in Google Drive ({DRIVE_RENDER_DIR})")
else:
    print(f"\n⚠ Warning: Frame count mismatch ({actual_frames} vs {TOTAL_FRAMES})")

# Optional direct browser download
from google.colab import files
print("Starting browser download of final video...")
files.download(FINAL_MASTER_MP4)


## Step 7: Post-Render Technical Check & Human Visual Review

This cell verifies the encoded stream and extracts a review contact sheet at the film's important transitions. It does **not** certify artistic quality automatically. Inspect the displayed frames for wave prominence, land contact, depth, gradual transformation, silhouette continuity, reflection placement, and the closing mood before treating the master as final.


In [ ]:
import os
import json
import subprocess
import shutil
import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Image as NotebookImage

print('=' * 76)
print('POST-RENDER TECHNICAL CHECK + VISUAL REVIEW CONTACT SHEET')
print('=' * 76)
if not os.path.exists(FINAL_MASTER_MP4):
    raise FileNotFoundError(f'Final master not found: {FINAL_MASTER_MP4}')

cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = actual_frames / fps if fps > 0 else 0.0
cap.release()
probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type', '-of', 'json', FINAL_MASTER_MP4], check=True, capture_output=True, text=True)
stream_types = [s.get('codec_type') for s in json.loads(probe.stdout).get('streams', [])]
checks = {'frame count': actual_frames == TOTAL_FRAMES, 'duration': abs(duration - DURATION) <= 0.05, 'picture size': (width, height) == (1920, 1080), 'audio stream': 'audio' in stream_types}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'} technical check: {label}")
print(f'  {actual_frames} frames | {fps:.3f} fps | {width}x{height} | {duration:.3f}s | streams={stream_types}')

review_points = [(24.0,'opening mushrooms'),(28.0,'mushroom 2 mature'),(31.9,'mushroom 2 hold ends'),(36.0,'mushroom 2 decay'),(40.1,'small decayed hold'),(54.0,'mushroom grounding'),(120.0,'mushroom hero hold'),(160.0,'arrival glint'),(166.0,'saucer enters'),(178.0,'saucer approach'),(182.5,'saucer landing hold'),(184.9,'landing hold end'),(198.0,'dawn departure'),(204.0,'saucer recedes'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(218.0,'forest before dinosaur entrance'),(232.0,'dinosaur entrance'),(260.0,'dinosaur forest hold'),(290.0,'pterodactyl passage'),(391.0,'caravan exit'),(397.0,'forest decay'),(400.0,'bare interval'),(406.0,'mushroom regrowth'),(414.0,'mushroom return'),(424.0,'Buddha transition'),(432.0,'CRT screens'),(440.0,'end credits')]
thumbs=[]
cap=cv2.VideoCapture(FINAL_MASTER_MP4)
for t,label in review_points:
    cap.set(cv2.CAP_PROP_POS_MSEC,t*1000); ok,bgr=cap.read()
    if not ok:
        print(f'Could not decode review frame at {t:.1f}s ({label})'); continue
    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    thumbs.append((t,label,Image.fromarray(rgb).resize((480,270),Image.Resampling.LANCZOS)))
cap.release()
cols=3; cell_w=480; cell_h=304; rows=(len(thumbs)+cols-1)//cols
sheet=Image.new('RGB',(cols*cell_w,rows*cell_h),(16,18,23)); draw=ImageDraw.Draw(sheet)
for i,(t,label,thumb) in enumerate(thumbs):
    x=(i%cols)*cell_w; y=(i//cols)*cell_h
    draw.text((x+8,y+5),f'{t:06.1f}s  {label}',fill=(236,224,195)); sheet.paste(thumb,(x,y+28))
review_sheet_path=os.path.join(OUTPUT_DIR,'final_deliverable_visual_review.jpg')
sheet.save(review_sheet_path,quality=92)
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_RENDER_DIR,exist_ok=True); shutil.copy2(review_sheet_path,os.path.join(DRIVE_RENDER_DIR,os.path.basename(review_sheet_path)))
print(f'Visual review sheet: {review_sheet_path} ({len(thumbs)} frames)')
display(NotebookImage(filename=review_sheet_path))
print('MANUAL ART REVIEW REQUIRED: this contact sheet does not certify composition, depth, transformation quality, wave stillness, or emotional impact.')
if not all(checks.values()): raise RuntimeError('Technical output check failed; inspect the master before distribution.')
